# 06 — Lakeflow Spark Declarative Pipeline Alternative

This notebook demonstrates how the **SNOW data product pipeline** (notebook 02) could alternatively be implemented as a **Lakeflow Spark Declarative Pipeline** (SDP).

### Why SDP for Data Mesh?
- **Auto Loader** for incremental ingestion from landing zones
- **Quality expectations** (`@dlt.expect`) as built-in data contracts
- **Automatic lineage** tracked in Unity Catalog
- **Schema evolution** handled declaratively
- **Streaming + batch** in a single pipeline

### Pipeline Structure
```
Landing Zone (JSON) → Bronze (raw) → Silver (cleansed) → Gold (service_health)
                       Auto Loader    Expectations        Aggregated product
```

**Target catalog:** `snow_product`  
**Owner:** ITSM Team (`itsm-team`)

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType

# Configuration — set via pipeline settings or widget defaults
LANDING = spark.conf.get(
    'landing_path',
    'abfss://raw@yourstorage.dfs.core.windows.net/servicenow'
)
TARGET_CATALOG = 'snow_product'

## Bronze Layer — Factory Pattern with Auto Loader

A single factory function creates streaming bronze tables for each ServiceNow entity. Auto Loader handles:
- Incremental file discovery (no re-processing)
- Automatic schema inference and evolution
- Source file tracking via `_metadata`

In [0]:
# ── BRONZE: Factory pattern for all ServiceNow entities ─────────────
def make_bronze(table_name):
    """Factory that creates a streaming bronze table for a ServiceNow entity."""
    @dlt.table(
        name=f'bronze_{table_name}',
        comment=f'Raw ServiceNow {table_name} records ingested via Auto Loader',
        table_properties={
            'quality': 'bronze',
            'data_product.owner': 'itsm-team',
            'data_product.sla': '24h',
            'pipelines.autoOptimize.zOrderCols': 'sys_id',
        },
    )
    def _():
        return (
            spark.readStream.format('cloudFiles')
            .option('cloudFiles.format', 'json')
            .option('cloudFiles.inferColumnTypes', 'true')
            .option('cloudFiles.schemaEvolutionMode', 'addNewColumns')
            .option('cloudFiles.schemaLocation', f'{LANDING}/_schemas/{table_name}')
            .load(f'{LANDING}/{table_name}/')
            .withColumn('_source_file', F.col('_metadata.file_path'))
            .withColumn('_loaded_at', F.current_timestamp())
        )
    return _

# Create bronze tables for all ITSM entities
for entity in ['incident', 'change_request', 'cmdb_ci', 'problem', 'task_sla']:
    make_bronze(entity)

## Silver Layer — Cleansed with Data Quality Expectations

Silver tables apply:
- **`@dlt.expect`** — Log violations but keep rows (monitoring)
- **`@dlt.expect_or_drop`** — Drop invalid rows (data contracts)
- Deduplication, type casting, and enrichment

These expectations serve as **automated data contracts** between the ITSM domain team (producer) and downstream consumers (CTO office, risk committee).

In [0]:
# ── SILVER: CMDB Configuration Items ─────────────────────────────────
@dlt.expect('valid_ci_name', 'name IS NOT NULL')
@dlt.expect('valid_status', "operational_status IN ('Operational', 'Non-Operational', 'Retired')")
@dlt.table(
    name='silver_cmdb_ci',
    comment='Cleansed CMDB CIs with criticality tiers and SLA targets',
    table_properties={'quality': 'silver', 'data_product.owner': 'itsm-team'},
)
def silver_cmdb_ci():
    return (
        dlt.read('bronze_cmdb_ci')
        .dropDuplicates(['sys_id'])
        .withColumn('tier_label',
            F.when(F.col('business_criticality') == '1', 'P1-Critical')
             .when(F.col('business_criticality') == '2', 'P2-High')
             .otherwise('P3-Standard'))
        .select('sys_id', 'name', 'sys_class_name', 'business_criticality',
                'tier_label', 'category', 'operational_status',
                'owned_by', 'support_group', '_loaded_at')
    )

In [0]:
# ── SILVER: Incidents with SLA breach detection ──────────────────────
@dlt.expect_or_drop('non_null_id', 'sys_id IS NOT NULL')
@dlt.expect_or_drop('valid_priority', "priority IN ('1', '2', '3', '4')")
@dlt.expect('valid_recovery', 'actual_recovery_minutes >= 0')
@dlt.table(
    name='silver_incident',
    comment='Cleansed incidents with recovery metrics and SLA breach flags',
    table_properties={'quality': 'silver', 'data_product.owner': 'itsm-team'},
)
def silver_incident():
    ci = (
        dlt.read('silver_cmdb_ci')
        .select('sys_id', 'name', 'tier_label', 'support_group')
        .withColumnRenamed('sys_id', 'ci_sys_id')
        .withColumnRenamed('name', 'ci_name')
    )

    inc = (
        dlt.read_stream('bronze_incident')
        .dropDuplicates(['sys_id'])
        .withColumn('opened_at', F.to_timestamp('opened_at'))
        .withColumn('resolved_at', F.to_timestamp('resolved_at'))
        .withColumn('actual_recovery_minutes',
            ((F.unix_timestamp('resolved_at') - F.unix_timestamp('opened_at')) / 60)
            .cast(IntegerType()))
    )

    return (
        inc.join(ci, inc['cmdb_ci'] == ci['ci_sys_id'], 'left')
        .withColumn('sla_breached', F.col('actual_recovery_minutes') > 240)
        .select(
            inc['sys_id'], inc['number'], inc['priority'],
            inc['opened_at'], inc['resolved_at'],
            'ci_name', 'tier_label', 'actual_recovery_minutes',
            'sla_breached', 'support_group', inc['_loaded_at']
        )
    )

In [0]:
# ── SILVER: Change Requests ──────────────────────────────────────────
@dlt.expect_or_drop('valid_type', "type IN ('normal', 'emergency', 'standard')")
@dlt.table(
    name='silver_change_request',
    comment='Cleansed change requests with success and emergency flags',
    table_properties={'quality': 'silver', 'data_product.owner': 'itsm-team'},
)
def silver_change_request():
    return (
        dlt.read_stream('bronze_change_request')
        .dropDuplicates(['sys_id'])
        .withColumn('opened_at', F.to_timestamp('opened_at'))
        .withColumn('is_emergency', F.col('type') == 'emergency')
        .withColumn('is_successful',
            F.col('close_code').isin(['successful', 'successful_with_issues']))
        .select('sys_id', 'number', 'type', 'state', 'risk',
                'cmdb_ci', 'opened_at', 'close_code',
                'is_emergency', 'is_successful', '_loaded_at')
    )

## Gold Layer — Service Health Data Product

The gold table aggregates incident and change metrics per application — the same output as `snow_product.gold.service_health` but built declaratively with full lineage and quality tracking.

In [0]:
# ── GOLD: Service Health (Data Product) ──────────────────────────────
@dlt.table(
    name='gold_service_health',
    comment='ITSM data product: per-application service health metrics including '
            'incident counts, MTTR, SLA compliance, and change success rates',
    table_properties={
        'quality': 'gold',
        'data_product.owner': 'itsm-team',
        'data_product.sla': '24h',
        'data_product.consumers': 'cto-office,risk-committee',
        'data_product.lineage': 'snow_product.silver.incident + snow_product.silver.change_request',
    },
)
def gold_service_health():
    incidents = dlt.read('silver_incident')
    changes = dlt.read('silver_change_request')

    inc_agg = (
        incidents.groupBy('ci_name')
        .agg(
            F.count('*').alias('total_incidents'),
            F.sum(F.when(F.col('priority') == '1', 1).otherwise(0)).alias('p1_incidents'),
            F.avg('actual_recovery_minutes').cast('int').alias('mttr_minutes'),
            F.round(
                F.sum(F.when(~F.col('sla_breached'), 1).otherwise(0)) / F.count('*') * 100, 1
            ).alias('sla_compliance_pct'),
        )
    )

    chg_agg = (
        changes.groupBy('cmdb_ci')
        .agg(
            F.count('*').alias('total_changes'),
            F.round(
                F.sum(F.when(F.col('is_successful'), 1).otherwise(0)) / F.count('*') * 100, 1
            ).alias('change_success_rate_pct'),
        )
    )

    return (
        inc_agg.join(chg_agg, inc_agg['ci_name'] == chg_agg['cmdb_ci'], 'left')
        .withColumn('health_score',
            F.round(F.col('sla_compliance_pct') * 0.6 + F.coalesce(F.col('change_success_rate_pct'), F.lit(100)) * 0.4, 1))
        .select(
            F.col('ci_name').alias('application_name'),
            'total_incidents', 'p1_incidents', 'mttr_minutes',
            'sla_compliance_pct', 'total_changes', 'change_success_rate_pct',
            'health_score',
            F.current_timestamp().alias('refreshed_at')
        )
    )

## Pipeline Deployment

To deploy this notebook as a Lakeflow Spark Declarative Pipeline:

1. **Workflows → Pipelines → Create Pipeline**
2. Set **source** to this notebook
3. Set **target catalog** = `snow_product`, **target schema** = `sdp`
4. Set **pipeline configuration**: `landing_path` = your ADLS landing zone path
5. Click **Start** for a full refresh or enable **continuous** for streaming

### Key Advantages over Batch Notebooks

| Feature | Batch Notebooks (02) | SDP Pipeline (this) |
|---|---|---|
| Incremental ingestion | Manual `MERGE INTO` | Auto Loader built-in |
| Quality enforcement | Custom SQL checks | `@dlt.expect` declarative |
| Lineage | TBLPROPERTIES (manual) | Automatic via UC |
| Schema evolution | Manual `ALTER TABLE` | `schemaEvolutionMode` |
| Error recovery | Re-run from scratch | Checkpoint-based resume |
| Monitoring | Custom quality log | Built-in event log + metrics |